# TSFresh-Features aus rohem, PCA- und DyCA-gefiltertem TEP-Signal

Umsetzung des zweiten TODO-Punkts aus `LazyClassifier_PCA_DyCA.ipynb`: statt der
**Eigenwerte** werden hier **TSFresh-Features** direkt aus dem Zeitsignal gewonnen —
einmal aus dem ungefilterten Signal, einmal aus dem PCA-projizierten und einmal aus
dem DyCA-projizierten. Anschließend Feature Selection auf die besten `top_k` Features
und LazyClassifier.

## Die 11 Konfigurationen

| Konfiguration | Kanäle | Projektion |
|---|---|---|
| `raw` | 52 | keine — die 52 Prozessvariablen direkt |
| `pca_4 … pca_12` | 4, 6, 8, 10, 12 | PCA pro Run, Scores der ersten *n* Hauptkomponenten |
| `dyca_m2_n4 … dyca_m6_n12` | 4, 6, 8, 10, 12 | DyCA pro Run, `amplitudes` (n, time) |

Zusammen 52 + 40 + 40 = **132 Kanäle pro Run**. Die DyCA-Paare sind n = 2m; das ist
exakt der Grenzfall der Bedingung `m >= n - m`, die das `dyca`-Paket erzwingt.

## Vorgehen (dreistufig)

**Phase A — Extraktion + Selektion auf TRAIN.** Je Konfiguration werden alle
TSFresh-Features auf allen Trainingsruns berechnet (`EfficientFCParameters`,
~780 Features je Kanal), daraus per Relevanztabelle die besten `top_k` ausgewählt.
Danach wird die volle Matrix sofort verworfen und nur `top_k` behalten.

**Phase B — Testset.** `tsfresh.feature_extraction.settings.from_columns()` übersetzt
die ausgewählten Featurenamen zurück in `kind_to_fc_parameters`. Auf dem Testset
werden deshalb **nur diese Features** berechnet — um Größenordnungen billiger als die
volle Extraktion. Die Selektion sieht das Testset nie.

**Phase C — LazyClassifier** je Konfiguration, dann Vergleich über alle 11.

## Laufzeit und Speicher — bitte vor dem Start lesen

Werte aus einem vollständigen Durchlauf auf dieser Maschine (8 Kerne,
`EfficientFCParameters`, `n_jobs=8`, alle 500 Runs je Fault): **30,2 Zeitreihen/s**.
Phase A dauerte **14,1 h** (davon allein 5,4 h für die 52 Rohkanäle), Phase B
1,4 h, Phase C 22 min — zusammen rund **16 h**.

Die Roh-Konfiguration erzeugt **40 404 Features pro Run** — als `float32` sind das
1,7 GB für die Trainingsmatrix. Deshalb: Feature-Matrizen als `float32`,
Konfigurationen strikt nacheinander, Relevanztests in Spaltenblöcken. Das
*projizierte Signal* geht dagegen als `float64` in TSFresh (Begründung im
Abschnitt „Projektionen").

**Der Cache macht den Lauf unterbrechbar.** Jeder Chunk landet als Pickle in
`cache_dir`; ein Neustart überspringt alles bereits Berechnete. Abbrechen ist also
gefahrlos.

> **`smoke_test` steht in der Konfigurationszelle.** Mit `True` läuft die komplette
> Pipeline in wenigen Minuten auf wenigen Runs durch — zum Prüfen, dass alles
> funktioniert. Für den echten Lauf auf `False` (am besten in tmux/über Nacht).
> Die Caches beider Modi liegen in getrennten Verzeichnissen.

## Code-Struktur

Der gesamte Maschinenraum liegt im Paket [`tep/`](tep) und wird von allen
TEP-Notebooks geteilt. Dieses Notebook enthält nur noch, was es von
seinen Geschwistern unterscheidet: die Konfiguration, die Liste der
Projektions-Specs und die Aufrufe.

| Modul | Inhalt |
|---|---|
| `tep/core.py` | Spaltennamen, Splits, Cutoffs, Vorverarbeitung, lineare Algebra — geteilt mit `tep.eigen` |
| `tep/plotting.py` | Confusion-Matrix-Darstellung, ebenfalls geteilt |
| `tep/tsfresh/config.py` | `PipelineConfig` — alle Stellschrauben |
| `tep/tsfresh/projections.py` | Registry der Verfahren: `raw`, `pca`, `dyca`, `dpca`, `cva`, `ica`, `dycvda` |
| `tep/tsfresh/features.py` | Chunk-Cache-Extraktion, Feature-Ranking |
| `tep/tsfresh/pipeline.py` | Phase A / B / C |
| `tep/tsfresh/reporting.py` | Vergleichstabellen und Balkenplot |
| `tep/tsfresh/confusion.py` | Confusion-Matrizen und ihre drei Plots |

Eine Projektion ist ein `Projector` mit drei Angaben: wie sie heisst (der
Name ist zugleich Cache-Praefix), wie ihre Kanaele heissen und wie sie
rechnet. Ein eigenes Verfahren kommt über `tep.tsfresh.register(...)`
dazu, ohne dass hier etwas angefasst werden muss.

> Die Rechnungen sind zeilengetreu aus der früheren Notebook-Fassung
> uebernommen — Projektionen und Cache-Praefixe wurden vor dem Umbau
> ueber alle Konfigurationen und beide Skalierungsmodi als bitidentisch
> nachgewiesen. Der vorhandene Chunk-Cache bleibt damit gueltig.


In [ ]:
# ============================================================
# Imports - der gemeinsame Unterbau steckt im Paket tep
# ============================================================
# Wird tep/**.py bearbeitet, muss der Kernel neu gestartet werden;
# alternativ die beiden autoreload-Zeilen aktivieren.
# %load_ext autoreload
# %autoreload 2

# numpy/pandas werden hier nicht gebraucht, stehen aber fuer eigene
# Auswertungen am Ende des Notebooks bereit.
import numpy as np
import pandas as pd

from tep.tsfresh import (Pipeline, PipelineConfig, plot_confusion_detail,
                         plot_recall, versions)

print(versions())

In [ ]:
# ============================================================
# Konfiguration - die EINZIGE Stelle, an der geschraubt wird
# ============================================================
# Alle nicht gesetzten Felder stehen auf den Defaults aus
# tep/tsfresh/config.py (top_k=100, fc_mode='efficient',
# chunk_runs=250, run_length=480, lc_cv_folds=5, ...).
# smoke_test=True gibt einen winzigen Probelauf in einem
# eigenen Cache-Ordner - ueberschreibt also nichts.

PCA_NS = [4, 6, 8, 10, 12]                            # PCA-Komponenten
DYCA_MN = [(2, 4), (3, 6), (4, 8), (5, 10), (6, 12)]  # DyCA (m, n), n = 2m

CFG = PipelineConfig(
    configs=([("raw",)]
             + [("pca", n) for n in PCA_NS]
             + [("dyca", m, n) for (m, n) in DYCA_MN]),
    label="PCA/DyCA",
    # Eigene Ergebnisdateien je Notebook-Familie: der Cache-Ordner ist
    # geteilt, die Schwester-Notebooks sollen sich nicht ueberschreiben.
    summary_csv="tsfresh_summary.csv",
    cm_pred_csv="tsfresh_cm_predictions.csv",
    scaling_mode="scaler",
)

## Rohdaten laden

Die Runs werden **einmal** in ein Dictionary `{(faultNumber, simulationRun): Array}`
gelesen und danach für alle 11 Konfigurationen wiederverwendet — die TEP-CSVs
(besonders `TEP_Faulty_Testing.csv` mit 3,4 GB) sollen nur ein Mal durch den Parser.

Speicher: mit `float32` und nur den benötigten Spalten sind das je ~1,05 GB für
Train und Test (beide Splits werden auf `run_length` gekürzt, siehe unten). Train
und Test werden **nie gleichzeitig** gehalten — Phase A braucht nur Train,
Phase B nur Test.

Sortiert wird pro Run (nicht global), weil ein globales `sort_values` über 10 Mio.
Zeilen eine komplette Kopie anlegen würde. DyCA braucht die zeitliche Ordnung für
die Ableitung, PCA nicht — sortiert wird trotzdem für beide, damit die Kanäle
identisch entstehen.

### `uniform_length` — bewusste Abweichung von den Eigenwert-Notebooks

Dort behält Fault 0 alle 500 Samples, während bei Fault 1–20 der Pre-Fault-Bereich
verworfen wird (480 Samples). Für die Eigenwerte ist das unkritisch — die erklärten
Varianzanteile sind praktisch längenunabhängig.

**Für TSFresh wäre es ein Kunstfehler.** `length` ist ein Feature (schon in
`MinimalFCParameters`), und `abs_energy`, `sum_values` oder `count_above_mean`
skalieren mit der Reihenlänge. Fault 0 wäre damit an der Länge allein perfekt
erkennbar — ein Artefakt der Vorverarbeitung, kein Prozesssignal. Der Klassifizierer
würde darauf hereinfallen und die Ergebnisse wären wertlos.

`uniform_length=True` wendet den Cutoff deshalb **auch auf Fault 0** an.

### `run_length` — gleiche Länge auch zwischen Train und Test

Damit wären die Runs *innerhalb* eines Splits gleich lang, zwischen den Splits aber
immer noch nicht: Training hat nach dem Cutoff 480 Samples, Test 800. Genau dieselben
längenabhängigen Features (`abs_energy`, `sum_values`, `absolute_sum_of_changes`,
`count_above_mean`, `number_peaks`, …) hätten dann in Train und Test **systematisch
verschiedene Größenordnungen** — ein Modell, das auf 480er-Werten trainiert wurde,
bekäme im Test rund das 1,7-Fache zu sehen. Das beschädigt jede Konfiguration
gleichermaßen und macht die Testbewertung wertlos.

`run_length = 480` kürzt deshalb **jeden** Run auf die ersten 480 Post-Fault-Samples.
Das entspricht in beiden Splits demselben physikalischen Fenster: 480 × 3 min = 24 h
nach Fehlereintritt. Vom Testset werden dadurch 320 Samples je Run verworfen — der
Preis dafür, dass Train- und Test-Features überhaupt dieselbe Bedeutung haben.

> Derselbe Effekt ist übrigens ein plausibler Mitgrund dafür, dass DyCA im
> Eigenwert-Notebook von 0,81 (CV auf Train) auf 0,39 (Test) einbricht — dort werden
> die Eigenwerte aus 480- bzw. 800-Sample-Fenstern berechnet.

## Projektionen: roh, PCA, DyCA

Alle drei Varianten bekommen **dieselbe Vorverarbeitung**, gesteuert über
`scaling_mode`. Damit unterscheidet sich zwischen den Konfigurationen wirklich nur
die Projektion.

### `scaling_mode` — hier hängt mehr dran, als es aussieht

`"global_mean"` ist die frühere Zeile aus den Eigenwert-Notebooks (vor
deren Scaler-Umstellung):
`X_scaled = X - X.mean()`. Das zieht einen **skalaren** Gesamtmittelwert über alle
52 Variablen und alle Zeitpunkte ab — die *spaltenweisen* Mittelwerte bleiben also
vollständig erhalten. Da die TEP-Variablen völlig verschiedene Größenordnungen haben
(`xmeas_2` ≈ 3663, `xmeas_1` ≈ 0,25), bleibt in jeder Spalte ein großer Gleichanteil
stehen.

Für **PCA** ist das folgenlos — `sklearn.PCA` zentriert intern ohnehin spaltenweise.
Für **DyCA** nicht: dessen Korrelationsmatrix `C0 = XᵀX/T` ist *unzentriert*, der
Gleichanteil dominiert sie also. Gemessen an einzelnen Runs (Faults 0, 1, 6, 13):

| Vorverarbeitung | std/\|mean\| der DyCA-Amplituden |
|---|---|
| `"global_mean"` | **1e-5 … 1e-3** |
| `"scaler"` | **0,3 … 28** |

Mit `"global_mean"` sind die DyCA-Kanäle, die in TSFresh gehen, numerisch also fast
reine Konstanten — die eigentliche Dynamik liegt drei bis fünf Größenordnungen
darunter. Das ist ein plausibler Hauptgrund dafür, dass DyCA sowohl hier als auch im
Eigenwert-Notebook schlecht abschneidet, und es ist genau das, was der dortige
TODO-Kommentar bereits vermutet.

`"scaler"` fittet einen `StandardScaler` auf `TEP_FaultFree_Training` (Normalbetrieb),
exakt wie in den anderen Notebooks. **Aktuell ist `"scaler"` eingestellt** —
im Gleichklang mit der Umstellung der Eigenwert-Notebooks auf
`scaler.transform(X)`. Der Cache-Ordner bekommt den Modus automatisch in
den Namen (`tsfresh_cache_scaler`); die alten `global_mean`-Ergebnisse in
`tsfresh_cache` bleiben unangetastet.

**PCA und DyCA werden beide pro Run gefittet.** DyCA kann gar nicht anders (es sucht
den Unterraum der Dynamik *dieses* Laufs), und für PCA hält das den Vergleich
symmetrisch — sonst würde man Methode und Fit-Modus vermischen.

**Vorzeichenkonvention.** Pro Run gefittete Achsen sind nur bis aufs Vorzeichen
bestimmt: PCA-Komponenten dürfen beliebig gespiegelt werden, DyCA-Amplituden ebenso
(SVD). Für die Eigenwerte in den bestehenden Notebooks war das egal — sie sind
vorzeicheninvariant. TSFresh-Features sind es **nicht**: `mean`, `skewness` oder die
Steigung von `linear_trend` kippen mit. Ohne Konvention wäre ein Teil der Features
reines Rauschen. `fix_signs=True` dreht deshalb jeden Kanal so, dass sein betragsmäßig
größter Wert positiv ist — deterministisch und für PCA wie DyCA identisch.

Bleibt die Einschränkung, dass pro Run gefittete Achsen in verschiedenen Runs
verschiedene physikalische Richtungen sind. Formstatistiken (Autokorrelation,
Spektrum, Entropie) bleiben vergleichbar; lageabhängige Features sind mit Vorsicht
zu lesen. Das ist der Preis der Symmetrie zu DyCA.

In [ ]:
# ============================================================
# Pipeline anlegen
# ============================================================
# Prueft die Specs (Rangbedingungen der Verfahren, doppelte Namen) und
# fittet bei scaling_mode="scaler" den StandardScaler auf dem
# Normalbetrieb. Danach steht der Umfang des Laufs im Klartext da.
pipe = Pipeline(CFG)
pipe.describe()

## Phase A — Extraktion und Selektion auf dem Trainingssatz

Je Konfiguration:

1. **Extrahieren** in Chunks à `chunk_runs` Runs. Jeder Chunk wird als `float32`-Pickle
   gecacht — ein Abbruch kostet höchstens den angefangenen Chunk.
2. **Selektieren:** Relevanztabelle in Spaltenblöcken (`block_cols`), damit die
   Roh-Konfiguration mit 40 404 Features nicht den Speicher sprengt.
3. **Reduzieren** auf `top_k` und die volle Matrix sofort freigeben.

**Zur Rangfolge.** Bei 10 500 Trainingsruns unterlaufen die p-Werte der
Signifikanztests reihenweise auf exakt 0.0 — nach p-Wert allein wären hunderte
Features gleichauf und die Auswahl der „besten 100" wäre willkürlich. Deshalb:
tsfresh liefert mit `n_significant` die Zahl der Klassen, die ein Feature signifikant
trennt (das ist bei `n_significant=1` genau das `relevant`-Kriterium), und innerhalb
gleicher `n_significant` wird nach dem **ANOVA-F-Wert** sortiert — eine stetige
Effektstärke, die nicht unterläuft.

Die Blockweise verschiebt die Benjamini-Hochberg-Korrektur minimal (sie sieht je Block
nur dessen p-Werte). Für eine *Rangfolge* ist das ohne Belang; darauf beruht die
Auswahl hier auch.

In [ ]:
# ============================================================
# PHASE A: Train extrahieren -> selektieren -> auf top_k reduzieren
# ============================================================
# Laeuft aus dem Chunk-Cache weiter, wenn schon Chunks vorhanden sind.
# Ergebnis liegt danach in pipe.train_top und pipe.top_names.
pipe.run_phase_a()

## Phase B — dieselben Features auf dem Testset

`from_columns()` übersetzt die ausgewählten Featurenamen zurück in ein
`kind_to_fc_parameters`-Dictionary. `extract_features` berechnet damit **nur** diese
Features — und auch nur auf den Kanälen, die in der Auswahl überhaupt vorkommen.
Deshalb ist Phase B um Größenordnungen billiger als Phase A (gemessen 1,4 h
gegenüber 14,1 h). Die Runs sind in beiden Splits gleich lang, weil `run_length`
auch die Testruns auf 480 Post-Fault-Samples kürzt.

Die Selektion hat ausschließlich Trainingsdaten gesehen — das Testset bleibt eine
unverzerrte Generalisierungsschätzung.

In [ ]:
# ============================================================
# PHASE B: Testset - nur die in Phase A ausgewaehlten Features
# ============================================================
# Ergebnis liegt danach in pipe.test_top.
pipe.run_phase_b()

## Phase C — LazyClassifier je Konfiguration

Alle 11 Konfigurationen laufen mit demselben Klassifizierer-Satz. Zwei Dinge sind
bewusst so gesetzt:

**Identische Run-Menge.** DyCA scheitert sporadisch und verliert Runs — und zwar
genau die Runs, an denen es numerisch scheitert. Würde man jede Konfiguration auf
ihrer eigenen Restmenge bewerten, bekämen die DyCA-Setups ein anderes (potenziell
leichteres) Testset als `raw` und die PCA-Setups. `restrict_to_common_runs=True`
schneidet deshalb alle auf die Runs zu, die in **allen** Konfigurationen vorhanden
sind.

**Macro-F1 als Kernzahl.** Balanced Accuracy ist der Macro-Recall und sieht Precision
nicht — sie belohnt Modelle, die eine Klasse als Sammelbecken missbrauchen. Macro-F1
bestraft das. Beides wird berichtet; lazypredicts eigene `F1 Score`-Spalte ist die
*gewichtete* Variante und deshalb hier nicht die Vergleichszahl.

**Modellauswahl über 5-fache CV — RandomForest bleibt der Hauptvergleich.**
Phase C läuft mit `lc_cv_folds = 5`: lazypredict kreuzvalidiert jedes Modell
zusätzlich auf den **Trainingsdaten** und liefert die Spalten `… CV Mean/Std`.
Die Auswahl „bestes Modell je Konfiguration" läuft darüber
(`BalancedAccCVMean`) und sieht das Testset nicht; berichtet werden weiterhin
die einmaligen Testwerte. Das alte Test-Maximum wird zum Vergleich mit
ausgegeben — es ist durch den Winner's Curse leicht optimistisch, und der
Abstand zwischen beiden zeigt, wie groß dieser Effekt hier ist.

Zwei Einschränkungen der CV:

- lazypredicts `F1 Score CV Mean` ist die **gewichtete** Variante; ein
  Macro-F1 aus der CV gibt es nicht. Selektionsmetrik ist deshalb
  `Balanced Accuracy CV Mean` — dieselbe Wahl wie `SELECT_METRIC` im
  Eigenwert-Notebook.
- Modelle **ohne `predict_proba`** (LinearSVC, Ridge, SGD, …) bekommen **keine**
  CV-Werte: lazypredict rechnet alle CV-Scorer gebündelt mit
  `error_score="raise"`, der ROC-AUC-Scorer braucht aber Wahrscheinlichkeiten —
  scheitert er, werden alle CV-Spalten dieses Modells geleert. Sie fallen damit
  aus der CV-Auswahl heraus (im Test-Maximum sind sie weiter dabei).

Der **RandomForest-Block** bleibt der belastbare Konfigurationsvergleich:
festes Modell, gar keine Auswahl.

> Kosten: die CV fittet jedes Modell fünfmal zusätzlich (Folds parallel,
> `n_jobs=-1`) — Phase C dauert grob das 2- bis 4-Fache. Die teuren Phasen A
> und B sind nicht betroffen und bleiben gecacht.

In [ ]:
# ============================================================
# PHASE C: LazyClassifier je Konfiguration
# ============================================================
# Schreibt die summary-CSV in den Cache-Ordner und legt sie zusaetzlich
# in pipe.summary ab. Die vollen Leaderboards stehen in
# pipe.leaderboards[name].
pipe.run_phase_c()

In [ ]:
# ============================================================
# Vergleich der Konfigurationen
# ============================================================
# Laeuft nach einem Kernel-Neustart auch OHNE Phase A/B/C: die summary
# liegt als CSV im Cache und wird von compare() nachgeladen. Vorher nur
# die Import-, Konfigurations- und Pipeline-Zelle ausfuehren.
cmp = pipe.compare()

In [ ]:
# ============================================================
# Plot: Hauptvergleich (RandomForest) + bestes Modell als Marker
# ============================================================
_ = pipe.plot_comparison(cmp)

## Confusion-Matrizen je Konfiguration

Dieselbe Auswertung wie am Ende von `LazyClassifier_PCA_DyCA.ipynb`, hier über
alle 11 Konfigurationen: **ein fester Modelltyp** — RandomForest, also der
Hauptvergleich von oben — wird pro Konfiguration auf dem **gesamten**
Trainingssatz gefittet und **einmal** auf dem echten Testset ausgewertet. Die
Unterschiede zwischen den Matrizen liegen damit allein an den Features.

**Warum neu gefittet wird.** Phase C berechnet die Vorhersagen zwar schon
(`predictions=True`), überschreibt `preds` aber je Konfiguration und hebt nichts
davon auf. Ein einzelner RF-Fit je Konfiguration auf `top_k = 100` Features
kostet Sekunden bis rund eine Minute — deutlich billiger, als Phase C mit ~25
Modellen zu wiederholen. Die Zelle gleicht am Ende gegen `tsfresh_summary.csv`
ab, ob sie die dortige RandomForest-Zeile reproduziert.

**Datenbasis identisch zu Phase C:** gemeinsame Runs
(`restrict_to_common_runs`), dieselbe NaN/inf-Behandlung, `StandardScaler` +
RandomForest wie lazypredict intern. Die gemeinsamen Runs werden hier neu
bestimmt — die Zelle läuft also auch ohne vorher gelaufenes Phase C, solange
`pipe.train_top`/`pipe.test_top` aus Phase A/B im Kernel liegen.

> Der `StandardScaler` in dieser Pipeline ist **nicht** `scaling_mode` — er
> standardisiert die 100 TSFresh-Features vor dem Klassifizierer (so macht es
> lazypredict intern) und läuft in beiden Skalierungsmodi gleich.
> `scaling_mode = "scaler"` betrifft die Vorverarbeitung **vor** der Projektion,
> ist also längst in den Features eingebacken, die hier ankommen.

**Darstellung.** Die Matrizen sind **zeilenweise normiert** (Zeilensumme = 1):
Zelle (i, j) ist der Anteil der wahren Klasse *i*, der als *j* vorhergesagt
wurde, die Diagonale also der Recall. Alle Panels teilen sich die Skala 0…1 und
sind dadurch direkt vergleichbar. Die absoluten Zählwerte stehen als
21×21-DataFrame in `cm.counts[name]`.

Die Vorhersagen werden als `tsfresh_cm_predictions.csv` im Cache abgelegt — hier
also in `tsfresh_cache_scaler/`, getrennt von der gleichnamigen Datei des
`global_mean`-Laufs in `tsfresh_cache/`. Beide lassen sich damit direkt
gegeneinanderlegen (siehe letzte Zelle). Nach einem Kernel-Neustart laufen die
Plotzellen darunter ohne Phase A/B/C; `refit=True` erzwingt die
Neuberechnung.

In [ ]:
# ============================================================
# Confusion-Matrizen: RandomForest je Konfiguration
# ============================================================
# refit=False nutzt den Vorhersage-Cache im Cache-Ordner; nach einem
# Kernel-Neustart laufen die Plotzellen darunter damit ganz ohne
# Phase A/B/C. refit=True fittet neu - noetig, wenn ueber estimator=...
# ein anderes Modell verglichen werden soll.
cm = pipe.confusion(refit=False)

In [ ]:
# ============================================================
# Plot: alle Confusion-Matrizen im Raster
# ============================================================
_ = pipe.plot_confusions(cm, ncols=4)

In [ ]:
# ============================================================
# Detail: eine Konfiguration gross + groesste Verwechslungen
# ============================================================
# focus=None waehlt die beste Konfiguration nach Macro-F1; sonst z.B.
# focus="raw". annot_min ist die Schwelle, ab der eine Zelle beschriftet
# wird, top_n die Laenge der Verwechslungsliste.
_, focus = plot_confusion_detail(cm, focus=None, annot_min=0.05, top_n=8)

cm.counts[focus]

In [ ]:
# ============================================================
# Recall je Fault-Klasse und Konfiguration
# ============================================================
recall_tab = plot_recall(cm)

recall_tab.round(3)

## Wo weitergemacht werden kann

- **Feature-Namen ansehen:** `pipe.top_names["pca_8"]` zeigt, *welche* TSFresh-Features
  eine Konfiguration ausgewählt hat — oft aufschlussreicher als der Score. Die
  vollständige Rangtabelle liefert `rank_features` (dort auch `p_min`).
- **`top_k` variieren:** die Auswahl ist gecacht, ein anderer Wert erzwingt aber eine
  neue Selektion. Die teure Extraktion in `cache_dir` bleibt gültig.
- **CV wieder abschalten:** `lc_cv_folds = 0` in der Konfigurationszelle nimmt
  die 5-fache Train-CV aus Phase C heraus und spart grob das 2- bis 4-Fache der
  Phase-C-Zeit; die Modellauswahl fällt dann auf das Test-Maximum zurück. Die
  teuren Phasen A/B sind nicht betroffen und bleiben gecacht.
- **Konfigurationen kombinieren:** `pd.concat([pipe.train_top["raw"], pipe.train_top["dyca_m4_n8"]], axis=1)`
  ergibt einen gemeinsamen Feature-Satz — analog zum PCA+DyCA-Fall im
  Eigenwert-Notebook.
- **Skalierung umstellen:** `scaling_mode` in der Konfigurationszelle von
  `"global_mean"` auf `"scaler"` — dann in den Eigenwert-Notebooks gleichziehen
  (dort ist es die Zeile `X_scaled = ...` mit dem `# TODO`). Der Cache-Ordner
  bekommt den Modus automatisch angehängt, alte Ergebnisse werden also nicht
  überschrieben.
- **Confusion-Matrizen:** das `cm`-Objekt aus der Zelle oben hält
  `cm.results` (Matrizen + Scores), `cm.counts` (absolute Zählwerte je
  Konfiguration) und `cm.recall_table()` (Recall je Klasse) bereit. Für
  ein anderes Modell `pipe.confusion(estimator=..., refit=True)` aufrufen
  — ohne `refit=True` wird der Vorhersage-Cache `tsfresh_cm_predictions.csv`
  weiterverwendet.
- **Skalierungsmodi vergleichen:** die Vorhersagen beider Läufe liegen als
  `tsfresh_cache/tsfresh_cm_predictions.csv` (`global_mean`) und
  `tsfresh_cache_scaler/tsfresh_cm_predictions.csv` (`scaler`) nebeneinander — beide laden,
  `cm.recall_table()` je Modus bauen und die Differenz ansehen zeigt, welche
  Fault-Klassen die Standardisierung tatsächlich rettet.

- **Eigenes Verfahren ergänzen:** ein neuer `Projector` wird über
  `tep.tsfresh.register("mein_verfahren", Projector(...))` eingetragen und
  ist danach als Spec `("mein_verfahren", …)` in `CFG.configs` nutzbar —
  an der Pipeline selbst muss dafür nichts geändert werden. Vorbild sind
  die sieben Einträge in `tep/tsfresh/projections.py`; ein `Projector`
  braucht nur `name`, `channels` und `apply`.